# 1: Data Ingestion
**Pipeline Stage:** Ingest raw data from multiple simulated sources

This notebook demonstrates:
- Batch ingestion from a CSV file (primary source)
- Simulated REST API ingestion
- Simulated real-time/streaming ingestion via a Kafka-like generator
- Data schema validation on arrival
- Saving raw ingested data to SQLite (the data lake layer)

In [1]:
# ─── Install dependencies (run once) ───────────────────────────────────────
# !pip install pandas sqlalchemy faker tqdm

In [2]:
import pandas as pd
import sqlite3
import json
import time
import random
import os
from datetime import datetime
from pathlib import Path

# ─── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR   = Path('..').resolve()
DATA_DIR   = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

CSV_PATH = DATA_DIR / 'online_shoppers_intention.csv'
DB_PATH  = OUTPUT_DIR / 'pipeline.db'

print(f"Source CSV : {CSV_PATH}")
print(f"SQLite DB  : {DB_PATH}")

Source CSV : C:\Stuffs\University Of New Haven\MSDS\Second Semester\Distributed and Scalable Data Engineering\Final\Project\Scripts\pipeline_project\pipeline\data\online_shoppers_intention.csv
SQLite DB  : C:\Stuffs\University Of New Haven\MSDS\Second Semester\Distributed and Scalable Data Engineering\Final\Project\Scripts\pipeline_project\pipeline\outputs\pipeline.db


## 1.1  Source A — Batch CSV Ingestion

In [3]:
def ingest_csv(path: Path) -> pd.DataFrame:
    """Load the CSV, add ingestion metadata, and return a DataFrame."""
    df = pd.read_csv(path)
    df['ingestion_source']    = 'csv_batch'
    df['ingestion_timestamp'] = datetime.utcnow().isoformat()
    df['row_id'] = range(len(df))
    print(f"[CSV] Loaded {len(df):,} rows × {df.shape[1]} columns")
    return df

df_batch = ingest_csv(CSV_PATH)
df_batch.head(3)

[CSV] Loaded 12,330 rows × 21 columns


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue,ingestion_source,ingestion_timestamp,row_id
0,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,...,1,1,1,1,Returning_Visitor,False,False,csv_batch,2026-05-04T01:36:23.593446,0
1,0,0.0,0,0.0,2,64.0,0.0,0.1,0.0,0.0,...,2,2,1,2,Returning_Visitor,False,False,csv_batch,2026-05-04T01:36:23.593446,1
2,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,...,4,1,9,3,Returning_Visitor,False,False,csv_batch,2026-05-04T01:36:23.593446,2


## 1.2  Source B — Simulated REST API Ingestion
Real-world scenario: an e-commerce platform exposes a paginated `/sessions` endpoint.  
We simulate pagination over the same data to demonstrate the pattern.

In [4]:
def simulate_api_fetch(df_source: pd.DataFrame, page: int, page_size: int = 100) -> dict:
    """Simulate a paginated JSON API response."""
    start = page * page_size
    end   = start + page_size
    chunk = df_source.iloc[start:end].copy()
    return {
        'status': 200,
        'page': page,
        'total_pages': len(df_source) // page_size,
        'data': chunk.to_dict(orient='records')
    }

api_records = []
NUM_API_PAGES = 5   # pull first 5 pages to demo

for page in range(NUM_API_PAGES):
    response = simulate_api_fetch(df_batch, page=page)
    for rec in response['data']:
        rec['ingestion_source']    = 'simulated_api'
        rec['ingestion_timestamp'] = datetime.utcnow().isoformat()
    api_records.extend(response['data'])

df_api = pd.DataFrame(api_records)
print(f"[API] Fetched {len(df_api):,} rows across {NUM_API_PAGES} pages")
df_api.head(2)

[API] Fetched 500 rows across 5 pages


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue,ingestion_source,ingestion_timestamp,row_id
0,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,...,1,1,1,1,Returning_Visitor,False,False,simulated_api,2026-05-04T01:36:23.649833,0
1,0,0.0,0,0.0,2,64.0,0.0,0.1,0.0,0.0,...,2,2,1,2,Returning_Visitor,False,False,simulated_api,2026-05-04T01:36:23.649833,1


## 1.3  Source C — Simulated Kafka Stream (Real-time)
We generate synthetic new sessions row-by-row to mimic a Kafka consumer.

In [5]:
MONTHS  = ['Jan','Feb','Mar','Apr','May','June','Jul','Aug','Sep','Oct','Nov','Dec']
VTYPES  = ['Returning_Visitor','New_Visitor','Other']

def generate_kafka_event(row_id: int) -> dict:
    """Simulate a single real-time shopper session event from a Kafka topic."""
    return {
        'row_id': row_id,
        'Administrative':             random.randint(0, 15),
        'Administrative_Duration':    round(random.uniform(0, 300), 2),
        'Informational':              random.randint(0, 10),
        'Informational_Duration':     round(random.uniform(0, 200), 2),
        'ProductRelated':             random.randint(0, 50),
        'ProductRelated_Duration':    round(random.uniform(0, 3000), 2),
        'BounceRates':                round(random.uniform(0, 0.2), 4),
        'ExitRates':                  round(random.uniform(0, 0.2), 4),
        'PageValues':                 round(random.uniform(0, 100), 2),
        'SpecialDay':                 round(random.choice([0,0,0,0.2,0.4,0.6,0.8,1.0]), 2),
        'Month':                      random.choice(MONTHS),
        'OperatingSystems':           random.randint(1, 8),
        'Browser':                    random.randint(1, 13),
        'Region':                     random.randint(1, 9),
        'TrafficType':                random.randint(1, 20),
        'VisitorType':                random.choice(VTYPES),
        'Weekend':                    random.choice([True, False]),
        'Revenue':                    random.random() < 0.155,   # ~15.5% base rate
        'ingestion_source':           'kafka_stream',
        'ingestion_timestamp':        datetime.utcnow().isoformat()
    }

NUM_STREAM_EVENTS = 200
stream_records = [generate_kafka_event(i) for i in range(NUM_STREAM_EVENTS)]
df_stream = pd.DataFrame(stream_records)
print(f"[Kafka] Received {len(df_stream):,} streaming events")
df_stream.head(2)

[Kafka] Received 200 streaming events


,row_id,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,...,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue,ingestion_source,ingestion_timestamp
0,0,8,133.91,8,78.64,28,2635.94,0.1212,0.0687,87.36,...,Sep,7,7,3,14,Other,False,False,kafka_stream,2026-05-04T01:36:23.701428
1,1,12,95.61,9,113.15,15,1020.41,0.1711,0.1188,10.43,...,Sep,5,6,5,7,New_Visitor,False,False,kafka_stream,2026-05-04T01:36:23.701428


## 1.4  Schema Validation

In [6]:
REQUIRED_COLUMNS = [
    'Administrative','Administrative_Duration','Informational','Informational_Duration',
    'ProductRelated','ProductRelated_Duration','BounceRates','ExitRates','PageValues',
    'SpecialDay','Month','OperatingSystems','Browser','Region','TrafficType',
    'VisitorType','Weekend','Revenue','ingestion_source','ingestion_timestamp'
]

def validate_schema(df: pd.DataFrame, name: str):
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    null_counts = df[REQUIRED_COLUMNS].isnull().sum()
    print(f"\n[VALIDATE] {name}")
    print(f"  Rows             : {len(df):,}")
    print(f"  Missing columns  : {missing or 'None'}")
    print(f"  Null counts (top): {null_counts[null_counts>0].to_dict() or 'None'}")

validate_schema(df_batch,  'CSV Batch')
validate_schema(df_api,    'Simulated API')
validate_schema(df_stream, 'Kafka Stream')


[VALIDATE] CSV Batch
  Rows             : 12,330
  Missing columns  : None
  Null counts (top): None

[VALIDATE] Simulated API
  Rows             : 500
  Missing columns  : None
  Null counts (top): None

[VALIDATE] Kafka Stream
  Rows             : 200
  Missing columns  : None
  Null counts (top): None


## 1.5  Store Raw Data to SQLite (Raw Data Lake)

In [7]:
# Combine all sources into one raw landing table
# Keep only batch + stream for the pipeline (API rows are a subset of batch)
df_raw = pd.concat([df_batch, df_stream], ignore_index=True)
df_raw['Revenue'] = df_raw['Revenue'].astype(int)   # SQLite-friendly
df_raw['Weekend'] = df_raw['Weekend'].astype(int)

conn = sqlite3.connect(DB_PATH)
df_raw.to_sql('raw_sessions', conn, if_exists='replace', index=False)
conn.close()

print(f"Wrote {len(df_raw):,} rows to table 'raw_sessions' in {DB_PATH}")

Wrote 12,530 rows to table 'raw_sessions' in C:\Stuffs\University Of New Haven\MSDS\Second Semester\Distributed and Scalable Data Engineering\Final\Project\Scripts\pipeline_project\pipeline\outputs\pipeline.db


In [8]:
# Quick verification read-back
conn = sqlite3.connect(DB_PATH)
verify = pd.read_sql('SELECT ingestion_source, COUNT(*) as n FROM raw_sessions GROUP BY ingestion_source', conn)
conn.close()
print(verify.to_string(index=False))

ingestion_source     n
       csv_batch 12330
    kafka_stream   200


## Summary
| Source | Rows | Method |
|---|---|---|
| CSV Batch | 12,330 | `pd.read_csv` |
| Simulated API | 500 (demo pages) | Paginated JSON fetch |
| Kafka Stream | 200 | Event generator |

All ingested data is stored in **`outputs/pipeline.db → raw_sessions`**.  